In [ ]:
import os

import arxiv

# Construct the default API client.
client = arxiv.Client()

# Search for the 10 most recent articles matching the keyword "quantum."
search = arxiv.Search(
    query="ti:quantum AND ti:annealing AND ti:optimization",
    max_results=5,
    sort_by=arxiv.SortCriterion.SubmittedDate,
)

In [8]:
import json
import socket
from datetime import datetime
from logging import getLogger

import requests

log = getLogger(__name__)


class DiscordWebhook:
    """DiscordのWebhook URLとチャンネルを指定して、メッセージを送信するクラス"""

    def __init__(self, url: str):
        """DiscordのWebhook URLとチャンネルを指定して初期化する。

        Args:
            url (str): DiscordのWebhook URL
            channel (str): メッセージを送信するDiscordのチャンネル名（例: "#general"）
        """
        self.url = url

        self.hostname = socket.gethostname()
        self.icon_emoji = ":desktop_computer:"
        self.current_time = datetime.now().strftime("%Y/%m/%d-%H時%M分")

    def send(
        self, title: str, message: str, footer: bool = True, footer_text: str = ""
    ) -> None:
        """Discordにメッセージを送信する。

        Args:
            title (str): メッセージのタイトル
            message (str): メッセージの内容
            footer (bool): フッターを表示するかどうか
        """
        footer_data = (
            {}
            if not footer
            else {
                "text": self.current_time + "\n" + footer_text,
                "icon_url": "https://cdn.discordapp.com/embed/avatars/2.png",
            }
        )
        payload = {
            "username": str(self.hostname),
            "embeds": [
                {
                    "title": title,
                    "description": message,
                    "color": 0x00FF00,
                    "footer": footer_data,
                }
            ],
        }
        response = requests.post(
            self.url,
            data=json.dumps(payload),
            headers={"Content-Type": "application/json"},
        )
        # エラーが出たらプリント
        if response.status_code == 204:
            log.info("メッセージを送信しました")
        else:
            log.error(f"エラーが発生しました: {response.status_code}")

In [ ]:
WEBHOOK_URL = os.environ.get("DISCORD_WEBHOOK_URL")
discord_bot = DiscordWebhook(url=WEBHOOK_URL)

In [ ]:
from deep_translator import GoogleTranslator

translator = GoogleTranslator(source="en", target="ja")

In [16]:
message_content = "📚 **本日のarXiv新着関連論文** 📚\n\n"
for result in client.results(search):
    message_content += f"🔹 **{result.title}**\n"
    message_content += f"🔗 URL: {result.entry_id}\n"
print(message_content)

📚 **本日のarXiv新着関連論文** 📚

🔹 **A Penalty-Free Pipeline for Direct Quantum-Annealer Portfolio Optimization**
🔗 URL: http://arxiv.org/abs/2605.17628v1
🔹 **Multi-Objective Optimization by Quantum-Annealing-Inspired Algorithms**
🔗 URL: http://arxiv.org/abs/2604.26477v2
🔹 **Variational and Annealing-Based Approaches to Quantum Combinatorial Optimization**
🔗 URL: http://arxiv.org/abs/2603.19117v1
🔹 **Adaptive Encoding Strategy for Quantum Annealing in Mixed-Variable Engineering Optimization**
🔗 URL: http://arxiv.org/abs/2603.17506v1
🔹 **Binary Latent Protein Fitness Landscapes for Quantum Annealing Optimization**
🔗 URL: http://arxiv.org/abs/2603.17247v1



In [14]:
for r in client.results(search):
    print(r.title)
    print(r.summary)
    print(r.entry_id)
    discord_bot.send(
        title=r.title,
        message=translator.translate(
            "【" + r.title + "】\n \n" + r.summary + "\n" + r.entry_id
        ),
        footer_text=r.entry_id,
    )
    print()

A Penalty-Free Pipeline for Direct Quantum-Annealer Portfolio Optimization
Direct quantum-annealer portfolio optimization is commonly formulated as a penalty-encoded QUBO and submitted to D-Wave hardware. We show that this standard formulation fails on current devices and identify the structural reason: the cardinality penalty contributes a dense rank-one term proportional to the all-ones matrix that makes the logical interaction graph complete regardless of the covariance structure. On Pegasus and Zephyr, chain-break fractions reach 83 percent at N equal to 24 and 92 percent at N equal to 49, producing no feasible samples. Attempting to fix this through topology-aware sparsification reveals a second problem: any sparsifier that removes off-diagonal entries also dilutes the cardinality constraint, so raw samples remain infeasible even when chains no longer break, and an ablation shows that for structurally favorable cases such as betting with settlement-graph priors the classical feasi